# Verificación del Entorno de Trabajo

**Análisis de Datos con Python — Axial Press**

Ejecuta este notebook antes de empezar el curso para comprobar que tu entorno está correctamente configurado.

▶️ **Cómo usarlo:** `Kernel → Restart & Run All` (o `Ejecutar todo`)

In [ ]:
import sys
import importlib
from pathlib import Path

# ── Utilidades de verificación ───────────────────────────────────────────────

OK  = "✓"
ERR = "✗"
errores = []

def check(condicion, msg_ok, msg_err):
    if condicion:
        print(f"  {OK}  {msg_ok}")
    else:
        print(f"  {ERR}  {msg_err}")
        errores.append(msg_err)

## 1 · Python

In [ ]:
print("Python")
version = sys.version_info
check(
    version >= (3, 10),
    f"Python {version.major}.{version.minor}.{version.micro}",
    f"Python {version.major}.{version.minor}.{version.micro} — se requiere 3.10 o superior"
)

## 2 · Librerías

In [ ]:
import re
from packaging.version import Version

# ── Leer versiones desde requirements.txt (fuente de verdad) ────────────────
IMPORT_A_PIP = {
    "sklearn": "scikit-learn",
    "PIL":     "Pillow",
    "cv2":     "opencv-python",
}

def _parse_requirements(path="requirements.txt"):
    """Devuelve dict {nombre_pip: version_minima_str} ignorando python y comentarios."""
    reqs = {}
    patron = re.compile(r"^([A-Za-z0-9_\-]+)\s*(?:==|>=|~=)\s*([\d.]+)")
    try:
        for linea in Path(path).read_text().splitlines():
            linea = linea.split("#")[0].strip()
            m = patron.match(linea)
            if m:
                nombre, version = m.group(1).lower(), m.group(2)
                if nombre != "python":
                    reqs[nombre] = version
    except FileNotFoundError:
        pass
    return reqs

# Solo librerías que realmente se usan en el libro
IMPORTS_A_VERIFICAR = [
    "numpy", "pandas", "matplotlib", "seaborn", "sklearn",
    "plotly", "streamlit", "joblib", "requests", "openpyxl",
]

reqs = _parse_requirements()

print("Librerías")
for nombre_import in IMPORTS_A_VERIFICAR:
    nombre_pip = IMPORT_A_PIP.get(nombre_import, nombre_import).lower()
    minima = reqs.get(nombre_pip, "0.0.0")
    instruccion = f"pip install {IMPORT_A_PIP.get(nombre_import, nombre_import)}>={minima}"
    try:
        mod = importlib.import_module(nombre_import)
        ver_str = getattr(mod, "__version__", "0.0.0")
        instalada = Version(ver_str)
        check(
            instalada >= Version(minima),
            f"{nombre_import} {ver_str}",
            f"{nombre_import} {ver_str} — se requiere >={minima}  →  {instruccion}"
        )
    except ImportError:
        check(False, "", f"{nombre_import} no instalado  →  {instruccion}")

## 3 · Datasets

In [ ]:
DATASETS = [
    ("UT01", "iris.csv"),
    ("UT02", "california_housing.csv"),
    ("UT03", "titanic.csv"),
    ("UT04", "air_quality_uci.csv"),
    ("UT05", "used_cars_raw.csv"),
    ("UT06", "house_prices.csv"),
    ("UT07", "winequality-red.csv"),
    ("UT08", "emp_attrition.csv"),
    ("UT09", "adult_income.csv"),
    ("UT10", "telco_churn.csv"),
]

print("Datasets")
data_dir = Path("data/raw")
for ut, fichero in DATASETS:
    ruta = data_dir / fichero
    check(
        ruta.exists(),
        f"{ut}  {fichero}",
        f"{ut}  {fichero} — no encontrado en {ruta}"
    )

## 4 · Resultado final

In [ ]:
print()
if not errores:
    print("══════════════════════════════════════════════")
    print("  Todo listo. Puedes empezar las prácticas.  ")
    print("══════════════════════════════════════════════")
else:
    print("══════════════════════════════════════════════")
    print(f"  {len(errores)} problema(s) encontrado(s):")
    print("══════════════════════════════════════════════")
    for e in errores:
        print(f"  {ERR}  {e}")
    print()
    print("Consulta el apartado 'Instalación del entorno'")
    print("en el Apéndice A del libro o en ENTORNOS.md del repositorio.")